#### Purpose of this notebook

This notebook contains the conversion of salary data to \$/year and store the corresponding dataset. It uses the dataset generated with vectors of dim 30 as embedding representation of the 'description' field.

#### Setup environment and libraries

In [ ]:
import os

In [8]:
# For Colab environment
#from google.colab import drive
#drive.mount('/content/drive')


In [2]:
# Change directory
# folder_path = "/content/drive/My Drive/Web Analytics/Final project scraped data/"
folder_path =r"C:\Users\adeli\Documents 4-Q1\Web Analytics\Final Project GitHub\web-analytics-project-repo"
os.chdir(folder_path)

In [3]:
# Check
current_dir = os.getcwd()
print(current_dir)

C:\Users\adeli\Documents 4-Q1\Web Analytics\Final Project GitHub\web-analytics-project-repo


In [ ]:
# Libraries to use
# Standard libraries
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)   # no column truncation
pd.set_option("display.max_rows", None)      # no row truncation
pd.set_option("display.width", None)         # no line-wrapping
pd.set_option("display.max_colwidth", None)
import numpy as np
import ast
import matplotlib.pyplot as plt
import seaborn as sns
# Specific libraries
import json
from pathlib import Path
import re

#### Transform dataset: Salary conversion to $/hour

In this notebook, as we have already established that the regression in \$/hour fares better than in \$/year (better performance), we perform only the conversion of salaries to \$/hour, and using the embeddings dim 30, to check whether they allow to a better regression performance than embeddings dim 300.

In [ ]:
# To reload after volunteer elimination and with embeddings dim 30 in description
file_name= "linkedin_description_30_vectors.csv"
df=pd.read_csv(file_name)
print(f"Data has been recovered from csv file.")

Data has been recovered from csv file.


In [10]:
print("\n====== Check NAs ======")
print(df.isna().sum().to_frame(name="NA_Counts"))


====== Check NAs ======
                   NA_Counts
offer_id                   0
title                      0
company_name               0
city                     408
state_code                 0
seniority_level            0
job_type                   0
job_function               5
industry                   1
description                0
salary_min              3673
salary_max              3673
salary_unit             3676
salary                  3673
searched_position          0
searched_location          0
applications               0
nltk_lemmas                0


In [11]:
from importlib import reload
import utilities
reload(utilities)

<module 'utilities' from 'C:\\Users\\adeli\\Documents 4-Q1\\Web Analytics\\Final Project GitHub\\web-analytics-project-repo\\utilities.py'>

In [12]:
from utilities import salary_conversion

In [17]:
# Need to map each job_type to hours_a_week, modify if needed
hours_map = {
    "Full-time": 40,
    "Part-time": 20,
    "Internship": 40,
    "Contract": 40,
    "Temporary": 40,
    "Other": 40,
}

# Create hours_a_week column
df["hours_a_week"] = df["job_type"].map(hours_map)

In [18]:
df.columns

Index(['offer_id', 'title', 'company_name', 'city', 'state_code',
       'seniority_level', 'job_type', 'job_function', 'industry',
       'description', 'salary_min', 'salary_max', 'salary_unit', 'salary',
       'searched_position', 'searched_location', 'applications', 'nltk_lemmas',
       'hours_a_week'],
      dtype='object')

In [19]:
# Check that now all entries with salary_min not NaN have salary_unit not NaN
mask_check = df["salary_min"].notna() & df["salary_unit"].isna()
num_check = mask_check.sum()
print(f"Number of entries with salary_min not NaN and salary_unit NaN: {num_check}")

Number of entries with salary_min not NaN and salary_unit NaN: 3


In [23]:
df[mask_check].drop(columns=["description", "nltk_lemmas"]).head(5)

,offer_id,title,company_name,city,state_code,seniority_level,job_type,job_function,industry,salary_min,salary_max,salary_unit,salary,searched_position,searched_location,applications,hours_a_week
115,115,Software Engineer,Scientific Research Corporation,San Diego,CA,Mid-Senior level,Full-time,Information Technology and Engineering,Engineering Services,127650.0,160000.0,NaN,"$127,650.00/daily - $160,000.00/daily",Software Engineer,San Diego,54 applicants,40
1348,1350,Systems Engineer IV,Scientific Research Corporation,San Diego,CA,Mid-Senior level,Full-time,Information Technology and Engineering,Engineering Services,146800.0,244650.0,NaN,"$146,800.00/daily - $244,650.00/daily",Systems Engineer,San Diego,26 applicants,40
6892,6894,Sr Network Engineer,Scientific Research Corporation,San Diego,CA,Mid-Senior level,Full-time,Information Technology and Engineering,Engineering Services,111100.0,185100.0,NaN,"$111,100.00/daily - $185,100.00/daily",Network Engineer,National City,Over 200 applicants,40


In [24]:
# Issue to correct: some entries have salary_unit as NaN, need to correct them.
# After examination, their unit is 'year', so we correct them.
df.loc[df["salary_unit"].isna(), "salary_unit"] = "year"

In [25]:
# Check again
mask_check = df["salary_min"].notna() & df["salary_unit"].isna()
num_check = mask_check.sum()
print(f"Number of entries with salary_min not NaN and salary_unit NaN: {num_check}")

Number of entries with salary_min not NaN and salary_unit NaN: 0


In [30]:
# Another issue detected: some entries have salary_unit as hour, but salary_max> 1000
mask_140= (df["salary_unit"] == "hour") & (df["salary_max"] > 140)
df_hour_over_140 = df[mask_140]
print(len(df_hour_over_140))
df_hour_over_140.drop(columns=["description", "nltk_lemmas"])

8


,offer_id,title,company_name,city,state_code,seniority_level,job_type,job_function,industry,salary_min,salary_max,salary_unit,salary,searched_position,searched_location,applications,hours_a_week
1678,1680,Senior Firmware Engineer,Kforce Inc,Cypress,CA,Associate,Contract,Engineering and Information Technology,IT Services and IT Consulting,124000.0,140000.0,hour,"$124,000.00/hr - $140,000.00/hr",Embedded Systems Engineer,Anaheim,Be among the first 25 applicants,40
3218,3220,"Field Service Engineer- Sacramento, CA",Avante Health Solutions,Sacramento,CA,Mid-Senior level,Full-time,Information Technology,Medical Equipment Manufacturing,150000.0,180000.0,hour,"$150,000.00/hr - $180,000.00/hr",Biomedical Engineer,Elk Grove,Be among the first 25 applicants,40
3302,3304,Ultrasound Field Service Engineer,Avante Health Solutions,Los Angeles,CA,Entry level,Full-time,Information Technology,Medical Equipment Manufacturing,75000.0,95000.0,hour,"$75,000.00/hr - $95,000.00/hr",Biomedical Engineer,Anaheim,Be among the first 25 applicants,40
6205,6207,Electrical Engineer II,CROWNMAZE LTD,El Dorado Hills,CA,Mid-Senior level,Full-time,Engineering and Information Technology,Hospitality,85891.0,119195.0,hour,"$85,891.00/hr - $119,195.00/hr",Hardware Engineer,Roseville,Be among the first 25 applicants,40
6454,6456,Mechanical Engineer II,CROWNMAZE LTD,El Dorado Hills,CA,Mid-Senior level,Full-time,Engineering and Information Technology,Hospitality,88969.0,107091.0,hour,"$88,969.00/hr - $107,091.00/hr",Mechanical Engineer,Roseville,Be among the first 25 applicants,40
9497,9502,Data Engineer/Analyst,Advanced Structural Technologies,Oxnard,CA,Associate,Full-time,Analyst,Manufacturing,80000.0,105000.0,hour,"$80,000.00/hr - $105,000.00/hr",Data Engineer,Ventura,Over 200 applicants,40
10045,10050,Quality Engineer,Jobot,Pasadena,CA,Not Applicable,Full-time,Quality Assurance,IT Services and IT Consulting and Software Development,110000.0,150000.0,hour,"$110,000.00/hr - $150,000.00/hr",Industrial Engineer,West Hollywood,Be among the first 25 applicants,40
10237,10243,Quality Engineer,Jobot,Pasadena,CA,Not Applicable,Full-time,Quality Assurance,IT Services and IT Consulting and Software Development,110000.0,150000.0,hour,"$110,000.00/hr - $150,000.00/hr",Industrial Engineer,Pomona,Be among the first 25 applicants,40


In [31]:
# Update salary_unit to 'year' for those rows
df.loc[mask_140, "salary_unit"] = "year"

In [32]:
# Check again
mask_140= (df["salary_unit"] == "hour") & (df["salary_max"] > 140)
df_hour_over_140 = df[mask_140]
print(len(df_hour_over_140))

0


In [33]:
def convert_row_to_hour(row):
    min_sal = row["salary_min"]
    max_sal = row["salary_max"]
    unit_in = row["salary_unit"]      # 'year', nan, 'hour', 'month'
    hours   = row["hours_a_week"]     # take from the mapping defined above
    
    # If any of these are missing, keep NaNs
    if pd.isna(min_sal) or pd.isna(max_sal) or pd.isna(unit_in):
        return pd.Series(
            [np.nan, np.nan],
            index=["salary_min_hour", "salary_max_hour"]
        )

    # Use the salary conversion function (utilities.py)
    min_y, max_y = salary_conversion(min_sal, max_sal, unit_in, "hour", hours)
    return pd.Series(
        [min_y, max_y],
        index=["salary_min_hour", "salary_max_hour"]
    )

In [34]:
# Apply the function to create the new columns, and the average salary per hour
df[["salary_min_hour", "salary_max_hour"]] = df.apply(convert_row_to_hour, axis=1)
df["salary_avg_hour"] = df[["salary_min_hour", "salary_max_hour"]].mean(axis=1)


In [35]:
df.columns

Index(['offer_id', 'title', 'company_name', 'city', 'state_code',
       'seniority_level', 'job_type', 'job_function', 'industry',
       'description', 'salary_min', 'salary_max', 'salary_unit', 'salary',
       'searched_position', 'searched_location', 'applications', 'nltk_lemmas',
       'hours_a_week', 'salary_min_hour', 'salary_max_hour',
       'salary_avg_hour'],
      dtype='object')

In [36]:
print("\n====== Check NAs ======")
print(df.isna().sum().to_frame(name="NA_Counts"))


====== Check NAs ======
                   NA_Counts
offer_id                   0
title                      0
company_name               0
city                     408
state_code                 0
seniority_level            0
job_type                   0
job_function               5
industry                   1
description                0
salary_min              3673
salary_max              3673
salary_unit                0
salary                  3673
searched_position          0
searched_location          0
applications               0
nltk_lemmas                0
hours_a_week               0
salary_min_hour         3673
salary_max_hour         3673
salary_avg_hour         3673


In [37]:
# To save
file_name= "linkedin_converted_salaries_hour_30emb.csv"
df.to_csv(file_name, index=False)
print(f"Data has been saved to file: {file_name}")
# df.to_csv("linkedin_converted_salaries_hour.csv", index=False)

Data has been saved to file: linkedin_converted_salaries_hour_30emb.csv


#### Legacy code

In [ ]:
# Used to correct three entries that were outliers because they had salary_unit as day
# when the should have been year
ids_with_nan = [115, 1350, 6894] 
df.loc[df["offer_id"].isin(ids_with_nan), "salary_unit"] = "year"